# DA6401 Report - Section 2.10 Fashion-MNIST Transfer Challenge

Evaluate **exactly 3 configurations** on Fashion-MNIST, chosen from MNIST learnings.
The goal is to test transfer of architecture + optimizer + activation choices.


In [1]:
from __future__ import annotations

import json
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import wandb

import sys
ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from train import train_and_evaluate

print('Project root:', ROOT)


Project root: /Users/akshayambekar/Code/da6401_assignments/da6401_assignment_1


In [2]:
RUN_EXPERIMENT = False  # Set True only when you intentionally want to launch new W&B runs.

PROJECT = 'da6401_assignment_1'
ENTITY = 'da25s007-'
OUT_DIR = ROOT / 'src' / 'tmp'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Chosen from MNIST learnings:
# C1: Best MNIST family (RMSProp + tanh + compact network)
# C2: Strong MNIST baseline (RMSProp + ReLU + 3x128)
# C3: Runner-up optimizer family from optimizer showdown (NAG + ReLU + 3x128)
configs = [
    {
        'name': 'fm_cfg1_mnist_best_family',
        'dataset': 'fashion_mnist',
        'epochs': 15,
        'batch_size': 64,
        'loss': 'cross_entropy',
        'optimizer': 'rmsprop',
        'learning_rate': 0.0005,
        'weight_decay': 0.0,
        'num_layers': 2,
        'hidden_size': [128, 128],
        'activation': 'tanh',
        'weight_init': 'random',
        'seed': 42,
    },
    {
        'name': 'fm_cfg2_rmsprop_relu_3x128',
        'dataset': 'fashion_mnist',
        'epochs': 15,
        'batch_size': 64,
        'loss': 'cross_entropy',
        'optimizer': 'rmsprop',
        'learning_rate': 0.001,
        'weight_decay': 0.0,
        'num_layers': 3,
        'hidden_size': [128, 128, 128],
        'activation': 'relu',
        'weight_init': 'xavier',
        'seed': 42,
    },
    {
        'name': 'fm_cfg3_nag_relu_3x128',
        'dataset': 'fashion_mnist',
        'epochs': 15,
        'batch_size': 64,
        'loss': 'cross_entropy',
        'optimizer': 'nag',
        'learning_rate': 0.001,
        'weight_decay': 0.0,
        'num_layers': 3,
        'hidden_size': [128, 128, 128],
        'activation': 'relu',
        'weight_init': 'xavier',
        'seed': 42,
    },
]

if RUN_EXPERIMENT:
    results = []

    for cfg in configs:
        args = SimpleNamespace(
            **cfg,
            wandb_project=PROJECT,
            wandb_entity=ENTITY,
            wandb_mode='online',
            model_path=str(OUT_DIR / f"model_2_10_{cfg['name']}.npy"),
            config_path=str(OUT_DIR / f"config_2_10_{cfg['name']}.json"),
        )

        run = wandb.init(
            project=PROJECT,
            entity=ENTITY,
            name=f"report_2_10_{cfg['name']}",
            tags=['report', '2.10', 'fashion_mnist', cfg['optimizer'], cfg['activation']],
            config=vars(args),
        )

        out = train_and_evaluate(args, wandb_run_override=run)

        rec = {
            'config_name': cfg['name'],
            'run_id': run.id,
            'run_name': run.name,
            'architecture': f"{cfg['num_layers']}x{cfg['hidden_size'][0] if len(set(cfg['hidden_size']))==1 else cfg['hidden_size']}",
            'optimizer': cfg['optimizer'],
            'activation': cfg['activation'],
            'learning_rate': cfg['learning_rate'],
            'final_val_accuracy': float(out['final_metrics']['val']['accuracy']),
            'final_test_accuracy': float(out['final_metrics']['test']['accuracy']),
            'final_val_f1': float(out['final_metrics']['val']['f1']),
            'final_test_f1': float(out['final_metrics']['test']['f1']),
        }
        results.append(rec)

        print(
            rec['config_name'],
            '| run', rec['run_id'],
            '| val_acc', round(rec['final_val_accuracy'], 4),
            '| test_acc', round(rec['final_test_accuracy'], 4),
        )

    results = sorted(results, key=lambda x: x['final_test_accuracy'], reverse=True)

    labels = [f"{r['optimizer']}|{r['activation']}|{r['architecture']}" for r in results]
    vals = [r['final_test_accuracy'] for r in results]

    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.bar(labels, vals)
    ax.set_title('2.10: Fashion-MNIST test accuracy for 3 transfer configs')
    ax.set_xlabel('Configuration (optimizer|activation|architecture)')
    ax.set_ylabel('Test accuracy')
    ax.set_ylim(0.0, 1.0)
    ax.grid(True, axis='y', alpha=0.3)
    plt.xticks(rotation=15, ha='right')
    fig.tight_layout()

    bar_path = OUT_DIR / 'report_2_10_fashion_transfer_bar.png'
    fig.savefig(bar_path, dpi=160, bbox_inches='tight')
    plt.close(fig)

    summary_run = wandb.init(
        project=PROJECT,
        entity=ENTITY,
        name='report_2_10_summary',
        tags=['report', '2.10', 'fashion_mnist', 'summary'],
        config={'num_configs': 3, 'source': 'mnist_learnings_transfer'},
    )

    table = wandb.Table(
        columns=[
            'rank', 'config_name', 'run_id', 'architecture', 'optimizer', 'activation',
            'learning_rate', 'val_accuracy', 'test_accuracy', 'val_f1', 'test_f1'
        ]
    )
    for i, r in enumerate(results, start=1):
        table.add_data(
            i, r['config_name'], r['run_id'], r['architecture'], r['optimizer'], r['activation'],
            r['learning_rate'], r['final_val_accuracy'], r['final_test_accuracy'], r['final_val_f1'], r['final_test_f1']
        )

    summary_run.log(
        {
            'analysis/fashion_transfer_accuracy_bar': wandb.Image(str(bar_path)),
            'analysis/fashion_transfer_results_table': table,
        }
    )

    for i, r in enumerate(results, start=1):
        summary_run.summary[f'rank{i}/config_name'] = r['config_name']
        summary_run.summary[f'rank{i}/run_id'] = r['run_id']
        summary_run.summary[f'rank{i}/test_accuracy'] = r['final_test_accuracy']

    summary_run.finish()

    payload = {
        'summary_run_name': 'report_2_10_summary',
        'summary_run_id': summary_run.id,
        'results_sorted_by_test_accuracy': results,
        'best_config': results[0],
        'did_mnist_best_transfer': bool(results[0]['config_name'] == 'fm_cfg1_mnist_best_family'),
        'artifact_paths': {
            'bar_plot': str(bar_path.relative_to(ROOT)),
        },
    }

    out_json = OUT_DIR / 'report_2_10_fashion_transfer.json'
    out_json.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    print('Wrote', out_json)
else:
    print('RUN_EXPERIMENT=False -> no W&B calls made.')


RUN_EXPERIMENT=False -> no W&B calls made.


In [3]:
summary_path = ROOT / 'src' / 'tmp' / 'report_2_10_fashion_transfer.json'
if summary_path.exists():
    data = json.loads(summary_path.read_text())
    print('Summary file:', summary_path)
    print('Summary run:', data['summary_run_name'], data['summary_run_id'])
    print('Did MNIST-best config remain best on Fashion-MNIST?:', data['did_mnist_best_transfer'])
    print('\nRanking by Fashion-MNIST test accuracy:')
    for i, row in enumerate(data['results_sorted_by_test_accuracy'], start=1):
        print(
            f"{i}. {row['config_name']} ({row['optimizer']}, {row['activation']}, {row['architecture']})"
            f" -> test_acc={row['final_test_accuracy']:.4f}, val_acc={row['final_val_accuracy']:.4f}, run={row['run_id']}"
        )
else:
    print('No summary JSON yet. Run the experiment cell first.')


Summary file: /Users/akshayambekar/Code/da6401_assignments/da6401_assignment_1/src/tmp/report_2_10_fashion_transfer.json
Summary run: report_2_10_summary jmw2pe04
Did MNIST-best config remain best on Fashion-MNIST?: True

Ranking by Fashion-MNIST test accuracy:
1. fm_cfg1_mnist_best_family (rmsprop, tanh, 2x128) -> test_acc=0.8809, val_acc=0.8955, run=gw65c11v
2. fm_cfg3_nag_relu_3x128 (nag, relu, 3x128) -> test_acc=0.8637, val_acc=0.8788, run=rxwnqla5
3. fm_cfg2_rmsprop_relu_3x128 (rmsprop, relu, 3x128) -> test_acc=0.8633, val_acc=0.8790, run=xveoufkw
